<div style="text-align: center; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; padding: 20px;">
    <h1 style="font-size: 80px; margin-bottom: 0; color: #1879beff; font-style: italic; font-weight: bold;">
        Invictus
    </h1>
    <h2 style="font-size: 28px; margin-top: 5px; color: #F6F6F6; font-weight: 300; letter-spacing: 1px;">
        Detección de Anomalías Hídricas y Atribución Causal Inteligente
    </h2>
    <hr style="width: 60%; border: 0; border-top: 1px solid #ddd; margin: 20px auto;">
</div>

<div align="center">

<a href="https://python.org" style="margin: 0 5px;">
    <img src="https://img.shields.io/badge/Python-3.13+-3776AB?style=for-the-badge&logo=python&logoColor=white" alt="Python">
</a>
<a href="https://streamlit.io" style="margin: 0 5px;">
    <img src="https://img.shields.io/badge/Streamlit-Dashboard-FF4B4B?style=for-the-badge&logo=streamlit&logoColor=white" alt="Streamlit">
</a>
<a href="https://scipy.org" style="margin: 0 5px;">
    <img src="https://img.shields.io/badge/Fourier-Series-4cc9f0?style=for-the-badge&logo=scipy&logoColor=white" alt="Fourier">
</a>
<a href="https://scikit-learn.org" style="margin: 0 5px;">
    <img src="https://img.shields.io/badge/Random%20Forest-Scikit--Learn-f39c12?style=for-the-badge&logo=scikit-learn&logoColor=white" alt="Random Forest">
</a>
<a href="https://apps.sentinel-hub.com/eo-browser/" style="margin: 0 5px;">
    <img src="https://img.shields.io/badge/Sentinel--2-NDVI-52b788?style=for-the-badge&logo=satellite&logoColor=white" alt="Sentinel-2">
</a>
<a href=".bases.pdf" style="margin: 0 5px;">
    <img src="https://img.shields.io/badge/Hackathon-2026-e74c3c?style=for-the-badge" alt="License">
</a>

<br><br>

<p style="width: 80%; font-size: 16px; line-height: 1.6; color: #AAA; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
    <strong>INVICTUS</strong> cruza series temporales de facturación hídrica con clima, vegetación satelital, datos turísticos y calendario festivo para revelar patrones de consumo anómalos. Su modelo híbrido detecta desviaciones y realiza atribuciones causales (fugas, estacionalidad climática, turismo), ofreciendo visibilidad absoluta sobre la demanda de la ciudad.
</p>

</div>

--- 

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append(os.getcwd())

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd

from src.model import ModeloFisico
from src.features.preprocessor import WaterPreprocessor
from src.config import DatasetKeys, Paths, FeatureConfig, get_logger

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")

# Inicializar configuración de rutas
Paths.init_project()
logger = get_logger("Notebook_Final")

## 1. Pipeline de Ingesta y Armonización de Datos (Fase 1)

El preprocesamiento orquesta la **convergencia de flujos de datos heterogéneos**. En esta etapa, el sistema realiza una limpieza profunda y un enriquecimiento mediante *feature engineering* avanzado, integrando en el conjunto de datos **AMAEM** métricas climáticas, índices de vegetación satelital, turismo (INE/GVA) y calendario festivo.

### Single Source of Truth (SSOT)
Para garantizar la robustez del software y evitar inconsistencias en el nombrado de variables, implementamos una capa de abstracción basada en la clase `DatasetKeys`.

* **Estandarización:** Referenciamos todas las entidades del dataset mediante constantes unificadas.
* **Trazabilidad:** Aseguramos que cada transformación, desde la carga hasta la atribución causal de SHAP, sea consistente.
* **Mantenibilidad:** El desacoplamiento entre el nombre físico de la columna en el CSV/SQL y su uso en el código permite actualizaciones de esquema sin riesgo de rotura (*breaking changes*).

In [ ]:
# Ejecutamos el orquestador global de procesamiento
df_scaled, df_not_scaled, scalers = WaterPreprocessor.process_all_data()

df_not_scaled_dom = df_not_scaled[df_not_scaled[DatasetKeys.USO] == 'DOMESTICO'].copy()

print(f"Dataset final DOMÉSTICO (features originales): {df_not_scaled_dom.shape}")
df_not_scaled_dom.sample(5)

## 2. Modelado Híbrido Físico-Estocástico (Fase 2)

Utilizamos `ModeloFisico.process()` para generar las predicciones basadas en la huella hídrica pura del barrio (**Fourier** de 2º orden) y el impacto exógeno estimado (**Random Forest**).

In [ ]:
# Lista de variables exógenas
feature_names = list(FeatureConfig.PIPELINE_FEATURES.keys())

logger.info("Entrenando Modelo Físico y Random Forest...")
df_final, rf_model, rf_features = ModeloFisico.process(df_not_scaled_dom, feature_names=feature_names)

print(f"Dataset consolidado con {df_final.shape[1]} columnas.")
display_cols = [DatasetKeys.FECHA, DatasetKeys.BARRIO, DatasetKeys.USO, 
                DatasetKeys.CONSUMO_RATIO, DatasetKeys.PREDICCION_FOURIER, 
                DatasetKeys.IMPACTO_EXOGENO, DatasetKeys.CONSUMO_FISICO_ESPERADO, 
                DatasetKeys.RESIDUO]
df_final[display_cols].head()


### Dinámica del Consumo: Real vs. Esperado vs. Anomalía
Ejemplo visual detallado para un barrio y uso específico, donde se puede observar el residuo entre lo que la base física predice, el impacto exógeno tolerado y el consumo real final.

In [ ]:
# 1. Configuración de estilo alineada con el Dashboard
sns.set_theme(style="whitegrid", rc={"axes.facecolor": "#f8f9fa", "grid.color": "#e9ecef"})

# 2. Selección de Caso de Estudio (Simulando la lógica de model.py)
barrio_sel = df_final[DatasetKeys.BARRIO].unique()[27]
uso_sel = 'DOMESTICO'
df_plot = df_final[(df_final[DatasetKeys.BARRIO] == barrio_sel) & 
                   (df_final[DatasetKeys.USO] == uso_sel)].copy()

df_plot[DatasetKeys.FECHA] = pd.to_datetime(df_plot[DatasetKeys.FECHA])
df_plot = df_plot.sort_values(DatasetKeys.FECHA)

# 3. Lógica Física: Recuperamos los parámetros de normalidad calculados en model.py
# El Z-Score se basa en la media y std del residuo de ESTE grupo específico
residuos_grupo = df_plot[DatasetKeys.RESIDUO]
mu = residuos_grupo.mean()
sigma = residuos_grupo.std()

plt.figure(figsize=(16, 7))

# 4. BANDA DE NORMALIDAD (Ground Truth de model.py)
# La banda no se centra en la predicción pura, sino en (Predicción + Sesgo Medio)
centro_normalidad = df_plot[DatasetKeys.CONSUMO_FISICO_ESPERADO] + mu
plt.fill_between(df_plot[DatasetKeys.FECHA], 
                 centro_normalidad - 1.5 * sigma, 
                 centro_normalidad + 1.5 * sigma, 
                 color='#52b788', alpha=0.12, label='Zona de Normalidad (Z < 1.5)', zorder=1)

# 5. LINEAS DE CONSUMO Y PREDICCIÓN
plt.plot(df_plot[DatasetKeys.FECHA], df_plot[DatasetKeys.CONSUMO_FISICO_ESPERADO], 
         'k--', linewidth=1.5, alpha=0.6, label='Predicción Híbrida (Fourier + RF)', zorder=2)

plt.plot(df_plot[DatasetKeys.FECHA], df_plot[DatasetKeys.CONSUMO_RATIO], 
         color='#0d1b2a', linewidth=2.5, label='Consumo Real', zorder=3)

# 6. ALERTAS (Mismo triaje que el Dashboard)
# Usamos directamente las etiquetas generadas por ModeloFisico
pintar = {
    '1_EXCESO_Grave':     {'c': '#c1121f', 's': 150, 'm': 'o', 'l': '🔴 Exceso Grave (Z > 2.5)'},
    '2_EXCESO_Moderado':  {'c': '#e76f51', 's': 110, 'm': 'o', 'l': '🟠 Exceso Moderado (Z > 2.0)'},
    '3_EXCESO_Leve':      {'c': '#f4a261', 's': 80,  'm': 'o', 'l': '🟡 Exceso Leve (Z > 1.5)'},
    '6_DEFECTO_Leve':     {'c': '#2d6a4f', 's': 80,  'm': 'X', 'l': '🟢 Defecto Leve (Z < -1.5)'},
    '5_DEFECTO_Moderado': {'c': '#1b4965', 's': 110, 'm': 'X', 'l': '🔵 Defecto Moderado (Z < -2.0)'},
    '4_DEFECTO_Grave':    {'c': '#0d1b2a', 's': 150, 'm': 'X', 'l': '⚫ Defecto Grave (Z < -2.5)'}
}

# Iteramos sobre los niveles de alerta para pintar los puntos
for nivel, estilo in pintar.items():
    mask = df_plot[DatasetKeys.ALERTA_NIVEL].str.contains(nivel, na=False)
    if any(mask):
        plt.scatter(df_plot.loc[mask, DatasetKeys.FECHA], df_plot.loc[mask, DatasetKeys.CONSUMO_RATIO], 
                    c=estilo['c'], s=estilo['s'], marker=estilo['m'], 
                    label=estilo['l'], edgecolors='white', zorder=5)

# 7. ESTÉTICA FINAL
plt.title(f'Auditoría INVICTUS: {barrio_sel} ({uso_sel})\nLógica de Producción: Fourier + RF + Z-Score Local', 
          fontsize=15, fontweight='bold', pad=20)
plt.ylabel('Ratio Consumo (m³/contrato)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=True, shadow=True)

# Ajuste de fechas
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 3. Detección de Fraude e Interpretabilidad IA (Fase 3)

Evolución y distribución histórica de las anomalías detectadas basadas en el Z-Score (`DatasetKeys.ALERTA_NIVEL`), mapeado en un sistema de triaje semafórico de 6 niveles.

In [ ]:
# Visualización de niveles de alerta
order_alertas = ['1_EXCESO_Grave', '2_EXCESO_Moderado', '3_EXCESO_Leve', 
                 'Normal', 
                 '6_DEFECTO_Leve', '5_DEFECTO_Moderado', '4_DEFECTO_Grave']

plt.figure(figsize=(10, 5))
ax = sns.countplot(y=DatasetKeys.ALERTA_NIVEL, data=df_final, 
                   order=order_alertas,
                   palette='coolwarm_r')
plt.title('Distribución Histórica de Alertas')
plt.xlabel('Conteo de Registros')
plt.ylabel('Nivel de Alerta')

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.show()


### Atribución Causal Automática (SHAP)
Análisis de interpretabilidad para descubrir las causas dominantes (temperatura, turismo, festivos, etc.) detrás de las alertas catalogadas como **Exceso Grave**.

In [ ]:
# Filtramos para analizar solo las causas de excesos graves
exceso_grave = df_final[df_final[DatasetKeys.ALERTA_NIVEL] == '1_EXCESO_Grave']

# Extraemos las columnas de porcentajes de SHAP del ground truth (FeatureConfig)
columnas_pct = list(set(FeatureConfig.CAUSAS_EXOGENAS.values())) # + [DatasetKeys.PCT_CAUSA_DESCONOCIDA]

if not exceso_grave.empty and all(col in exceso_grave.columns for col in columnas_pct):
    # Calculamos la importancia media
    causas_plot = exceso_grave[columnas_pct].mean().sort_values(ascending=True)
    
    # Limpiamos los nombres para la gráfica (quitamos 'pct_' y guiones bajos)
    etiquetas = [c.replace('pct_', '').replace('_', ' ').title() for c in causas_plot.index]
    
    # Paleta de colores oficial de producción (frío a caliente)
    colores_prod = ['#1b4965', '#2d6a4f', '#52b788', '#f4a261', '#e76f51', '#c1121f']
    colors = sns.color_palette(colores_prod, len(causas_plot))
    
    plt.figure(figsize=(10, 6))
    bars = plt.barh(etiquetas, causas_plot.values, color=colors, edgecolor='#0d1b2a', alpha=0.9)
    
    # Añadimos etiquetas de porcentaje en las barras
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
                 f'{width:.1f}%', va='center', fontweight='bold', color='#0d1b2a')

    plt.title('Atribución Causal Media (Alertas de Exceso Grave)\n¿Qué variables exógenas influyen más en el modelo?', 
              fontsize=14, fontweight='bold', color="#0d1b2a", pad=15)
    plt.xlabel('Influencia Ponderada en la Detección (%)', fontsize=12, fontweight='bold')
    plt.xlim(0, max(causas_plot.values) + 15)
    
    # Limpiar bordes
    sns.despine(left=True, bottom=True)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes alertas de '1_EXCESO_Grave' o faltan columnas PCT para generar la atribución causal.")

### Mapa Térmico de Tensión Hídrica (Matriz Z-Score)

Para complementar el análisis individual, esta visualización ofrece una **vista macro panorámica** de toda la ciudad de Alicante. Al representar el valor continuo de la anomalía (`DatasetKeys.Z_ERROR_FINAL`) en una matriz espaciotemporal (Barrio vs. Mes), podemos detectar rápidamente patrones sistémicos y correlaciones ocultas:

* 🔴 **Zonas Rojas (Tensión / Exceso):** Señalan clústeres de sobreconsumo. 
    * *Si coinciden verticalmente (mismo mes):* Indican un evento a nivel de ciudad (ej. ola de calor no registrada, festividad masiva).
    * *Si se mantienen horizontalmente (mismo barrio a lo largo del tiempo):* Apuntan a un problema estructural (ej. fraude sistemático, red deteriorada, alta concentración de viviendas turísticas ilegales).
* 🔵 **Zonas Azules (Defecto):** Indican caídas inusuales de consumo (ej. cortes de suministro prolongados, éxodo poblacional).
* ⚪ **Zonas Blancas (Normalidad):** El motor híbrido de IA ha explicado y predecido perfectamente el comportamiento del consumo, absorbiendo con éxito el impacto de los factores climáticos y turísticos.

In [ ]:
import re
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Función para ordenación natural (detecta números y los ordena matemáticamente)
def natural_sort_key(s):
    # Divide el texto en partes de texto y números, convirtiendo los números a int
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', str(s))]

# Agrupamos el Z-Score medio por Barrio y Fecha
pivot_z = df_final.pivot_table(index=DatasetKeys.BARRIO, 
                               columns=DatasetKeys.FECHA, 
                               values=DatasetKeys.Z_ERROR_FINAL, 
                               aggfunc='mean')

# ORDENAMOS LOS BARRIOS CON ORDENACIÓN NATURAL
barrios_ordenados = sorted(pivot_z.index, key=natural_sort_key)
pivot_z = pivot_z.loc[barrios_ordenados]

# Formateamos las columnas (fechas) para que se lean bien
pivot_z.columns = [pd.to_datetime(c).strftime('%Y-%m') for c in pivot_z.columns]

# Aumentamos el alto para acomodar todos los barrios sin problemas
plt.figure(figsize=(16, 12)) 

# Usamos el colormap divergente oficial (Azul defecto, Blanco normal, Rojo exceso)
cmap = sns.diverging_palette(250, 10, as_cmap=True)

# Creamos el heatmap forzando a que se muestren todas las etiquetas Y
ax = sns.heatmap(pivot_z, cmap=cmap, center=0, 
                 cbar_kws={'label': 'Z-Score (Desviación Normalizada)'},
                 linewidths=0.5, linecolor='white',
                 yticklabels=True)

plt.title('Mapa Térmico de Tensión Hídrica: Evolución del Z-Score por Barrio', 
          fontsize=16, fontweight='bold', color="#0d1b2a", pad=20)
plt.xlabel('Línea Temporal (Meses)', fontsize=12, fontweight='bold')
plt.ylabel('Barrios de Alicante', fontsize=12, fontweight='bold')

plt.xticks(rotation=45)
# Ajustamos el tamaño de la fuente de los barrios para una lectura limpia
plt.yticks(rotation=0, fontsize=9) 

plt.tight_layout()
plt.show()

<hr style="border: 0; border-top: 1px solid rgba(128, 128, 128, 0.4); margin: 30px 0;">

El uso de `DatasetKeys` junto con nuestra arquitectura garantiza que el notebook sea robusto ante cambios en el esquema de datos, manteniendo la integridad científica de las 3 fases del pipeline: **Preprocesamiento**, **Modelado Físico-Estocástico** y **Detección de Fraude**. Al aislar temporalmente el entrenamiento (2022-2023) de la predicción (2024), evitamos el *data leakage* asegurando que las alertas generadas sean genuinas.

<div align="center">
<br>

### Hecho con 💧 por el equipo INVICTUS
*Hackathon 2026 · Alicante, España*

</div>